# SEACO-6161 · Generic VACUUM cleanup

Works on **the workspace it runs in**: no workspace config, no `%run`, no audit tables. Import it into any
workspace (OPAS, ARC, ...) and run it on a **Single User** cluster. Built to finish in about **30 minutes**:

1. **Inventory (metadata only, no file listing):** every table in the chosen databases with its row count
   (Delta statistics), live size, live file count, last write, last VACUUM and the bytes written since then,
   grouped by database.
2. **Clone check:** finds shallow-clone sources. A VACUUM on a source deletes files its clones still read,
   so these are never VACUUMed.
3. **Candidates:** Delta tables written since their last VACUUM, ranked by bytes written since then (the old
   versions VACUUM can remove come from those writes), at most `max_tables`. Or set `vacuum_tables` to pick them.
   Tables vacuumed in the last `skip_vacuumed_days` days (20) are skipped and listed; set
   `apply_recent_vacuum_filter = false` to include them.
   Table folders are never listed or sized, which is what keeps the run short.
4. **VACUUM** with `RETAIN retain_hours HOURS`, printing rows before and after for each table. Freed space and
   deleted files come from VACUUM's own metrics in the table history (`VACUUM START` / `VACUUM END`), so the
   folders are not listed again.
   - `dry_run = true` (the default): runs `VACUUM … DRY RUN`. Nothing is deleted. It shows how many files would
     be deleted (Databricks lists at most 1,000), not their size; the size is known after the real run.
   - `dry_run = false`: needs `confirm_workspace` = this workspace's name, for example
     `dbw-opas-merone-pipeline-prod` (printed in the setup cell; looked up from `KNOWN_WORKSPACES` by workspace ID). Rows are
     counted by reading the data files, on the same table version, before and after. If they differ the run
     stops.
   - **`time_budget_min`:** no new table starts after this many minutes. A VACUUM that is already running
     finishes first, and a table with millions of small files can take hours on its own. Run again for the rest.
5. **Summary:** GiB freed in total and per database, and how many databases and tables were freed.

VACUUM never goes below 168 hours, and the notebook stops if `retentionDurationCheck` has been disabled.
Concurrent writes are safe: VACUUM only deletes files that have been unused for longer than the retention.

In [0]:
dbutils.widgets.text("databases", "", "01. Databases (comma list, empty = all)")
dbutils.widgets.text("vacuum_tables", "", "02. Only these tables (db.table, comma list)")
dbutils.widgets.text("exclude_tables", "", "03. Never VACUUM (db.table, comma list)")
dbutils.widgets.text("retain_hours", "720", "04. Retain hours (>= 168)")
dbutils.widgets.text("max_tables", "20", "05. Max tables to VACUUM")
dbutils.widgets.text("time_budget_min", "25", "06. Start no new table after N minutes")
dbutils.widgets.text("skip_vacuumed_days", "20", "07. Skip tables vacuumed in the last N days")
dbutils.widgets.dropdown("apply_recent_vacuum_filter", "true", ["true", "false"], "08. Apply the N-day filter")
dbutils.widgets.dropdown("clone_check_scope", "all", ["all", "selected"], "09. Clone check: all or selected dbs")
dbutils.widgets.dropdown("dry_run", "true", ["true", "false"], "10. Dry run")
dbutils.widgets.text("confirm_workspace", "", "11. Workspace name (needed when dry_run = false)")

In [0]:
import datetime as dt
import time
from concurrent.futures import ThreadPoolExecutor

from pyspark.sql import functions as F

GIB = 1024 ** 3
MIN_RETAIN_HOURS = 168
THREADS = 16
META_KEY = "spark.databricks.delta.optimizeMetadataQuery.enabled"

CATALOG = spark.sql("SELECT current_catalog()").first()[0]
WORKSPACE_URL = spark.conf.get("spark.databricks.workspaceUrl", "unknown")
WORKSPACE_ID = spark.conf.get("spark.databricks.clusterUsageTags.clusterOwnerOrgId", "unknown")

# Databricks does not expose the Azure workspace name to a notebook, so it is looked up by workspace ID
# (the number in adb-<id>.<n>.azuredatabricks.net). Add a workspace here the first time you run it there.
KNOWN_WORKSPACES = {
    "2440482787921388": "dbw-opas-merone-pipeline-prod",
    # "<ARC workspace ID>": "dbw-arc-merone-pipeline-prod",
}


def _workspace_name():
    if WORKSPACE_ID in KNOWN_WORKSPACES:
        return KNOWN_WORKSPACES[WORKSPACE_ID], "known workspace ID"
    try:  # some workspaces carry their name as a cluster tag
        import json as _json
        tags = {t["key"]: t["value"] for t in _json.loads(
            spark.conf.get("spark.databricks.clusterUsageTags.clusterAllTags", "[]"))}
        for k in ("WorkspaceName", "DatabricksWorkspaceName", "workspace_name", "workspace"):
            if tags.get(k):
                return tags[k], f"cluster tag {k}"
    except Exception:
        pass
    return f"workspace-{WORKSPACE_ID}", "not in KNOWN_WORKSPACES; add it there"


WORKSPACE_NAME, WORKSPACE_NAME_SOURCE = _workspace_name()
_ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
RUN_USER = _ctx.userName().get() if _ctx.userName().isDefined() else "unknown"

T_START = time.time()
_csv = lambda w: [x.strip() for x in dbutils.widgets.get(w).split(",") if x.strip()]
DATABASES_ARG = _csv("databases")
ONLY = {t.lower() for t in _csv("vacuum_tables")}
EXCLUDE = {t.lower() for t in _csv("exclude_tables")}
if ONLY and not DATABASES_ARG:
    DATABASES_ARG = sorted({t.split(".", 1)[0] for t in ONLY})
RETAIN_HOURS = int(dbutils.widgets.get("retain_hours"))
MAX_TABLES = int(dbutils.widgets.get("max_tables"))
BUDGET_S = float(dbutils.widgets.get("time_budget_min")) * 60
SKIP_DAYS = int(dbutils.widgets.get("skip_vacuumed_days"))
RECENT_FILTER = dbutils.widgets.get("apply_recent_vacuum_filter") == "true" and SKIP_DAYS > 0
CLONE_SCOPE = dbutils.widgets.get("clone_check_scope")
DRY_RUN = dbutils.widgets.get("dry_run") != "false"
CONFIRM = dbutils.widgets.get("confirm_workspace").strip()

_jvm = spark._jvm
_hconf = spark._jsc.hadoopConfiguration()


def human(n):
    if n is None:
        return "-"
    n = float(n)
    for unit in ("B", "KiB", "MiB", "GiB", "TiB", "PiB"):
        if abs(n) < 1024 or unit == "PiB":
            return f"{n:,.2f} {unit}"
        n /= 1024


def num(n):
    return f"{n:,}" if n is not None else "-"


def qname(db, table=None):
    return ".".join(f"`{p}`" for p in [CATALOG, db] + ([table] if table else []))


def list_databases():
    return sorted(r[0] for r in spark.sql(f"SHOW DATABASES IN `{CATALOG}`").collect())


def list_tables(db):
    try:
        rows = spark.sql(f"SHOW TABLES IN {qname(db)}").collect()
    except Exception as e:
        print(f"  ! SHOW TABLES {db}: {str(e)[:200]}")
        return []
    try:
        views = {r["viewName"] for r in spark.sql(f"SHOW VIEWS IN {qname(db)}").collect()}
    except Exception:
        views = set()
    return sorted(r["tableName"] for r in rows if not r["isTemporary"] and r["tableName"] not in views)


def elapsed():
    return f"{(time.time() - T_START) / 60:,.1f} min"


_WRITE_BYTES_KEYS = ("numOutputBytes", "numTargetBytesAdded", "numAddedBytes")


def history_info(q):
    """Last VACUUM and bytes written since then, from DESCRIBE HISTORY (reaches back logRetentionDuration,
    30 days by default). Writes since the last VACUUM are what turned into old versions."""
    rows = spark.sql(f"DESCRIBE HISTORY {q}").select("version", "timestamp", "operation", "operationMetrics").collect()
    vac = [r for r in rows if (r["operation"] or "").startswith("VACUUM")]
    last_vac = max((r["timestamp"] for r in vac), default=None)
    since = [r for r in rows if not (r["operation"] or "").startswith(("VACUUM", "OPTIMIZE"))
             and (last_vac is None or r["timestamp"] > last_vac)]
    written = 0
    for r in since:
        m = r["operationMetrics"] or {}
        written += next((int(m[k]) for k in _WRITE_BYTES_KEYS if m.get(k)), 0)
    return dict(last_vacuum=last_vac, writes_since=len(since), written_since=written,
                history_from=min((r["timestamp"] for r in rows), default=None))


def vacuum_metrics(q, after_version):
    """Files and bytes deleted by the VACUUM that ran after after_version, from its START / END history rows."""
    rows = (spark.sql(f"DESCRIBE HISTORY {q} LIMIT 20").select("version", "operation", "operationMetrics")
            .filter(F.col("version") > after_version).collect())
    start = next((r["operationMetrics"] or {} for r in rows if r["operation"] == "VACUUM START"), {})
    end = next((r["operationMetrics"] or {} for r in rows if r["operation"] == "VACUUM END"), {})
    to_int = lambda x: int(x) if x not in (None, "") else None
    return dict(files_to_delete=to_int(start.get("numFilesToDelete")),
                bytes_to_delete=to_int(start.get("sizeOfDataToDelete")),
                files_deleted=to_int(end.get("numDeletedFiles")))


def normalize(path):
    p = (path or "").strip()
    if p.startswith("dbfs:"):
        p = p[5:]
    return p.rstrip("/")


def is_under(path, parent):
    path, parent = normalize(path), normalize(parent)
    return bool(parent) and (path == parent or path.startswith(parent + "/"))

In [0]:
print(f"Workspace     : {WORKSPACE_NAME}  ({WORKSPACE_NAME_SOURCE})")
print(f"Workspace URL : {WORKSPACE_URL}  (ID {WORKSPACE_ID})")
print(f"Catalog       : {CATALOG}")
print(f"Run as        : {RUN_USER}")
print(f"Mode          : {'DRY RUN (nothing is deleted)' if DRY_RUN else 'REAL VACUUM'}")
print(f"Retain        : {RETAIN_HOURS} hours ({RETAIN_HOURS / 24:.0f} days of time travel kept)")
print(f"Candidates    : {'only ' + ', '.join(sorted(ONLY)) if ONLY else f'at most {MAX_TABLES} tables, ranked by bytes written since last VACUUM'}")
print(f"Time budget   : no new table after {BUDGET_S / 60:,.0f} min")
print(f"Recent VACUUM : {f'skip tables vacuumed in the last {SKIP_DAYS} days' if RECENT_FILTER else 'filter off (tables vacuumed recently are included)'}")

In [0]:
problems = []
if RETAIN_HOURS < MIN_RETAIN_HOURS:
    problems.append(f"retain_hours must be >= {MIN_RETAIN_HOURS}")
if spark.conf.get("spark.databricks.delta.retentionDurationCheck.enabled", "true").lower() != "true":
    problems.append("spark.databricks.delta.retentionDurationCheck.enabled has been set to false on this cluster")
if not DRY_RUN and CONFIRM != WORKSPACE_NAME:
    problems.append(f"dry_run = false needs confirm_workspace = {WORKSPACE_NAME}")
if problems:
    raise Exception("Safety gates failed:\n  - " + "\n  - ".join(problems))

DATABASES = DATABASES_ARG or list_databases()
missing = sorted(set(DATABASES) - set(list_databases()))
if missing:
    raise Exception(f"Databases not found in {CATALOG}: {', '.join(missing)}")

In [0]:
%scala
// Shallow-clone check. A table whose current files live OUTSIDE its own folder is a shallow clone; the
// folder it points into belongs to the SOURCE table, and a VACUUM on that source would break the clone.
// Input : temp view _clone_check_input(table_name, table_location)
// Output: temp view _clone_check_output(table_name, referenced_dir, n_files, error)
import spark.implicits._

def normalizeDbfs(p: String): String = {
  var s = Option(p).getOrElse("")
  if (s.startsWith("dbfs:")) s = s.substring(5)
  s.stripSuffix("/")
}

def findShallowCloneRefs(): Unit = {
  val input = spark.table("_clone_check_input").as[(String, String)].collect()
  val results = input.par.flatMap { case (name, location) =>
    try {
      val loc = normalizeDbfs(location)
      val outside = spark.read.format("delta").load(location).inputFiles
        .map(normalizeDbfs).filterNot(f => f.startsWith(loc + "/"))
      outside.groupBy(f => f.substring(0, math.max(f.lastIndexOf('/'), 0)))
        .map { case (dir, fs) => (name, dir, fs.length.toLong, null: String) }.toSeq
    } catch {
      case e: Throwable => Seq((name, null: String, 0L, String.valueOf(e.getMessage).take(300)))
    }
  }.seq.toSeq
  results.toDF("table_name", "referenced_dir", "n_files", "error").createOrReplaceTempView("_clone_check_output")
  println(s"clone check: ${input.length} Delta tables checked, ${results.count(_._2 != null)} external file groups, " +
          s"${results.count(_._4 != null)} errors")
}

## 1. Inventory: databases, tables, row counts and sizes (metadata only)

In [0]:
def detail(db, t):
    row = dict(database=db, table=t, format=None, location=None, live_bytes=None, live_files=None,
               last_modified=None, row_count=None, last_vacuum=None, writes_since=None, written_since=None,
               error=None)
    try:
        d = spark.sql(f"DESCRIBE DETAIL {qname(db, t)}").first().asDict()
        row.update(format=(d.get("format") or "").lower() or None, location=d.get("location"),
                   live_bytes=d.get("sizeInBytes"), live_files=d.get("numFiles"),
                   last_modified=d.get("lastModified"))
        if row["format"] == "delta":
            # count(*) on Delta is answered from the log statistics: no data is read.
            row["row_count"] = spark.sql(f"SELECT count(*) AS c FROM {qname(db, t)}").first()["c"]
            h = history_info(qname(db, t))
            row.update(last_vacuum=h["last_vacuum"], writes_since=h["writes_since"], written_since=h["written_since"])
    except Exception as e:
        row["error"] = str(e)[:300]
    return row


todo = [(db, t) for db in DATABASES for t in list_tables(db)]
if ONLY:
    todo = [(db, t) for db, t in todo if f"{db}.{t}".lower() in ONLY]
with ThreadPoolExecutor(THREADS) as pool:
    INVENTORY = list(pool.map(lambda x: detail(*x), todo))

INV_COLS = ["database", "table", "format", "row_count", "live_bytes", "live_files", "last_modified", "last_vacuum",
            "writes_since", "written_since", "location", "error"]
inventory_df = spark.createDataFrame(
    [tuple(r[c] for c in INV_COLS) for r in INVENTORY],
    "database string, table string, format string, row_count long, live_bytes long, live_files long, "
    "last_modified timestamp, last_vacuum timestamp, writes_since long, written_since long, location string, "
    "error string",
)

In [0]:
print(f"{len(INVENTORY):,} tables in {len(DATABASES)} databases, inventory done at {elapsed()}\n")
print(f"{'table':55s} {'format':7s} {'rows':>15s} {'live':>12s} {'files':>10s} {'last VACUUM':>12s} {'written since':>14s}")
for db in DATABASES:
    sub = sorted((r for r in INVENTORY if r["database"] == db), key=lambda r: r["table"])
    if not sub:
        continue
    print(f"\n== {db}: {len(sub)} tables, {num(sum(r['row_count'] or 0 for r in sub))} rows, "
          f"{human(sum(r['live_bytes'] or 0 for r in sub))} live, "
          f"{human(sum(r['written_since'] or 0 for r in sub))} written since last VACUUM")
    for r in sub:
        lv = r["last_vacuum"].strftime("%Y-%m-%d") if r["last_vacuum"] else ("never" if r["format"] == "delta" else "-")
        print(f"  {r['table']:53s} {(r['format'] or '-'):7s} {num(r['row_count']):>15s} {human(r['live_bytes']):>12s} "
              f"{num(r['live_files']):>10s} {lv:>12s} {human(r['written_since']):>14s}"
              + (f"  ! {r['error'][:80]}" if r["error"] else ""))
print(f"\nTotal: {len(INVENTORY)} tables, {human(sum(r['live_bytes'] or 0 for r in INVENTORY))} live")

In [0]:
display(inventory_df
        .withColumn("live_gib", F.round(F.col("live_bytes") / GIB, 2))
        .withColumn("written_since_gib", F.round(F.col("written_since") / GIB, 2))
        .select("database", "table", "format", "row_count", "live_gib", "live_files", "last_modified", "last_vacuum",
                "writes_since", "written_since_gib", "location", "error")
        .orderBy(F.desc_nulls_last("written_since_gib")))

## 2. Shallow-clone check and VACUUM candidates

In [0]:
clone_input = [(f"{r['database']}.{r['table']}", r["location"]) for r in INVENTORY
               if r["format"] == "delta" and r["location"]]
if CLONE_SCOPE == "all":
    # A clone in another database can point into a table selected here, so check every Delta table.
    def delta_location(db, t):
        try:
            d = spark.sql(f"DESCRIBE DETAIL {qname(db, t)}").first().asDict()
            return (f"{db}.{t}", d["location"]) if (d.get("format") or "").lower() == "delta" else None
        except Exception:
            return None
    others = [(db, t) for db in list_databases() if db not in DATABASES for t in list_tables(db)]
    with ThreadPoolExecutor(THREADS) as pool:
        clone_input += [x for x in pool.map(lambda x: delta_location(*x), others) if x]
spark.createDataFrame(clone_input, "table_name string, table_location string") \
    .createOrReplaceTempView("_clone_check_input")

In [0]:
%scala
findShallowCloneRefs()

In [0]:
clone_rows = spark.table("_clone_check_output").collect()
CLONE_ERRORS = [r for r in clone_rows if r["error"]]
CLONE_SOURCES = {}   # "db.table" of a source -> the clones that read its files
for ref in (r for r in clone_rows if r["referenced_dir"]):
    for r in INVENTORY:
        if r["format"] == "delta" and r["location"] and is_under(ref["referenced_dir"], r["location"]):
            CLONE_SOURCES.setdefault(f"{r['database']}.{r['table']}", set()).add(ref["table_name"])


def needs_vacuum(r):
    # Written since its last VACUUM (or never vacuumed) = there can be old versions to remove.
    return r["last_vacuum"] is None or (r["last_modified"] is not None and r["last_modified"] > r["last_vacuum"])


# Tables vacuumed in the last SKIP_DAYS days are left alone (turn off with apply_recent_vacuum_filter = false).
# DESCRIBE HISTORY timestamps are in the session time zone, so compare with the same clock.
NOW = spark.sql("SELECT current_timestamp() AS t").first()["t"]
RECENT_CUTOFF = NOW - dt.timedelta(days=SKIP_DAYS)
RECENTLY_VACUUMED = sorted(
    (r for r in INVENTORY if RECENT_FILTER and r["format"] == "delta" and r["last_vacuum"] is not None
     and r["last_vacuum"] >= RECENT_CUTOFF),
    key=lambda r: (r["database"], r["table"]))
_recent = {f"{r['database']}.{r['table']}" for r in RECENTLY_VACUUMED}


CANDIDATES = sorted(
    [r for r in INVENTORY
     if r["format"] == "delta" and not r["error"]
     and (ONLY or needs_vacuum(r))
     and f"{r['database']}.{r['table']}".lower() not in EXCLUDE
     and f"{r['database']}.{r['table']}" not in _recent
     and f"{r['database']}.{r['table']}" not in CLONE_SOURCES],
    key=lambda r: (-(r["written_since"] or 0), -(r["live_bytes"] or 0)),
)[:MAX_TABLES]

In [0]:
for src, clones in sorted(CLONE_SOURCES.items()):
    print(f"  SKIP clone source {src} (read by {', '.join(sorted(clones))})")
if RECENTLY_VACUUMED:
    print(f"  SKIP {len(RECENTLY_VACUUMED)} tables vacuumed in the last {SKIP_DAYS} days "
          f"(set apply_recent_vacuum_filter = false to include them):")
    for r in RECENTLY_VACUUMED:
        print(f"    {r['database'] + '.' + r['table']:70s} last VACUUM {r['last_vacuum']:%Y-%m-%d %H:%M}")
if CLONE_ERRORS:
    print(f"  ⚠️  {len(CLONE_ERRORS)} Delta tables could not be checked for clones, e.g. "
          f"{CLONE_ERRORS[0]['table_name']}: {CLONE_ERRORS[0]['error'][:120]}")
print(f"\n{len(CANDIDATES)} tables to VACUUM ({'dry run' if DRY_RUN else 'REAL'}), candidates ready at {elapsed()}:")
for r in CANDIDATES:
    print(f"  {r['database'] + '.' + r['table']:70s} {human(r['live_bytes']):>12s} live, {num(r['live_files'])} files, "
          f"{human(r['written_since'])} written since last VACUUM")

## 3. VACUUM, with row counts before and after

In [0]:
class RowCheckFailed(Exception):
    pass


def current_version(q):
    return spark.sql(f"DESCRIBE HISTORY {q} LIMIT 1").first()["version"]


def scan_count(q, version):
    """Row count of one table version, read from the data files (not the Delta statistics)."""
    old = spark.conf.get(META_KEY, "true")
    spark.conf.set(META_KEY, "false")
    try:
        return spark.sql(f"SELECT count(*) AS c FROM {q} VERSION AS OF {version}").first()["c"]
    finally:
        spark.conf.set(META_KEY, old)


spark.conf.set("spark.databricks.delta.vacuum.parallelDelete.enabled", "true")
RESULTS = []
for r in CANDIDATES:
    name, q = f"{r['database']}.{r['table']}", qname(r["database"], r["table"])
    res = dict(database=r["database"], table=r["table"], version=None, rows_before=None, rows_after=None,
               files_before=None, files_after=None, freed_bytes=None, files_listed=None, status=None,
               seconds=None, detail=None)
    if time.time() - T_START > BUDGET_S:
        res.update(status="skipped_time", detail=f"time budget of {BUDGET_S / 60:,.0f} min used; run again")
        RESULTS.append(res)
        continue
    t0 = time.time()
    try:
        res["version"] = v = current_version(q)
        if DRY_RUN:
            res["rows_before"] = res["rows_after"] = r["row_count"]   # from Delta statistics
            print(f"\n{name} @ version {v}  [{elapsed()}]\n  rows before   : {num(res['rows_before'])}")
            res["files_listed"] = spark.sql(f"VACUUM {q} RETAIN {RETAIN_HOURS} HOURS DRY RUN").count()
            res["status"] = "dry_run"
            print(f"  DRY RUN       : {'1,000+' if res['files_listed'] >= 1000 else num(res['files_listed'])} files "
                  f"to delete")
            print(f"  rows after    : {num(res['rows_after'])} (unchanged, nothing deleted)")
        else:
            res["rows_before"] = scan_count(q, v)
            print(f"\n{name} @ version {v}  [{elapsed()}]\n  rows before   : {num(res['rows_before'])}")
            spark.sql(f"VACUUM {q} RETAIN {RETAIN_HOURS} HOURS")
            m = vacuum_metrics(q, v)
            live_files = spark.sql(f"DESCRIBE DETAIL {q}").first()["numFiles"]
            res["freed_bytes"] = m["bytes_to_delete"]
            res["files_after"] = live_files
            deleted = m["files_deleted"] if m["files_deleted"] is not None else m["files_to_delete"]
            res["files_before"] = live_files + deleted if deleted is not None else None
            if m["bytes_to_delete"] is None:
                res["detail"] = "no VACUUM START metrics in history (spark.databricks.delta.vacuum.logging.enabled off?)"
            res["rows_after"] = scan_count(q, v)
            print(f"  rows after    : {num(res['rows_after'])}")
            print(f"  VACUUM        : {num(deleted)} files deleted, {human(res['freed_bytes'])} freed "
                  f"in {time.time() - t0:,.0f}s")
            if res["rows_after"] != res["rows_before"]:
                res["status"] = "row_check_failed"
                raise RowCheckFailed(f"DATA CHECK FAILED on {name} @ version {v}: "
                                     f"{res['rows_before']:,} rows before, {res['rows_after']:,} after")
            res["status"] = "ok"
    except RowCheckFailed as e:
        res["detail"] = str(e)
        res["seconds"] = time.time() - t0
        RESULTS.append(res)
        raise
    except Exception as e:
        res["status"], res["detail"] = "error", str(e)[:300]
        print(f"  ! {name}: {str(e)[:300]}")
    res["seconds"] = time.time() - t0
    RESULTS.append(res)

RES_COLS = ["database", "table", "version", "status", "rows_before", "rows_after", "files_before", "files_after",
            "freed_bytes", "files_listed", "seconds", "detail"]
results_df = spark.createDataFrame(
    [tuple(x[c] for c in RES_COLS) for x in RESULTS],
    "database string, table string, version long, status string, rows_before long, rows_after long, "
    "files_before long, files_after long, freed_bytes long, files_listed long, seconds double, detail string",
)

In [0]:
print(f"{'table':70s} {'version':>8s} {'status':14s} {'rows before':>15s} {'rows after':>15s} "
      f"{'files before':>13s} {'files after':>12s} {'freed':>12s}")
for x in RESULTS:
    print(f"{x['database'] + '.' + x['table']:70s} {num(x['version']):>8s} {x['status']:14s} "
          f"{num(x['rows_before']):>15s} {num(x['rows_after']):>15s} {num(x['files_before']):>13s} "
          f"{num(x['files_after']):>12s} {human(x['freed_bytes']):>12s}")

## 4. Summary

In [0]:
done = [x for x in RESULTS if x["status"] in ("ok", "dry_run")]
freed = [x for x in done if (x["freed_bytes"] or 0) > 0]
by_db = {}
for x in freed:
    d = by_db.setdefault(x["database"], dict(tables=0, freed=0))
    d["tables"] += 1
    d["freed"] += x["freed_bytes"]
total = sum(x["freed_bytes"] for x in freed)
word = "Would free (estimate)" if DRY_RUN else "Freed"
if DRY_RUN:
    print("Dry run: VACUUM only lists the files it would delete, so no size is shown. The real run reports it.\n")

print(f"==== Summary ({'DRY RUN, nothing deleted' if DRY_RUN else 'REAL VACUUM'}) on {WORKSPACE_NAME} ====")
print(f"Tables VACUUMed   : {len(done)} of {len(CANDIDATES)} candidates "
      f"({sum(x['status'] == 'error' for x in RESULTS)} errors, "
      f"{sum(x['status'] == 'skipped_time' for x in RESULTS)} left for the next run)")
print(f"Run time          : {elapsed()}")
print(f"Row check         : {'not run (dry run)' if DRY_RUN else ('all tables unchanged' if all(x['rows_before'] == x['rows_after'] for x in done) else 'MISMATCH')}")
print(f"{word:18s}: {total / GIB:,.2f} GiB ({human(total)})")
print(f"Databases freed   : {len(by_db)}")
print(f"Tables freed      : {len(freed)}")
for db, d in sorted(by_db.items(), key=lambda kv: -kv[1]["freed"]):
    print(f"  {db:50s} {d['tables']:>4} tables  {d['freed'] / GIB:>12,.2f} GiB")
if DRY_RUN and CANDIDATES:
    print(f"\nTo run for real: dry_run = false, confirm_workspace = {WORKSPACE_NAME}")

In [0]:
display(results_df
        .withColumn("freed_gib", F.round(F.col("freed_bytes") / GIB, 2))
        .select("database", "table", "status", "version", "rows_before", "rows_after", "files_before",
                "files_after", "freed_gib", "files_listed", "seconds", "detail"))

## Schedule as a monthly job

Import this notebook into each workspace and create the job from the YAML below (Workflows → Create job →
"Edit as YAML", or a Databricks Asset Bundle). Change `notebook_path` and the e-mail address for your workspace.

- **Schedule:** `0 0 3 28 * ?` runs at 03:00 on the **28th of every month** (every month has a 28th, so none is
  skipped).
- **The scheduled run is a dry run** (`dry_run: "true"`): it lists the candidate tables and the files VACUUM would
  delete, and deletes nothing. Review the run output, then use **Run now with different parameters** with
  `dry_run = false` and `confirm_workspace` = the workspace name printed in the setup cell (for example
  `dbw-opas-merone-pipeline-prod`).
- **E-mails:** success means a dry run is ready for review; failure means a cell errored or a row count changed
  (`row_check_failed`). The e-mail only links to the run. The candidate list is in the run output.
- **Cluster:** runtime 16.4 LTS (Scala 2.12: the clone check uses parallel collections, which Scala 2.13 does not
  include), `SINGLE_USER` access mode (needed for `spark._jvm`, the Scala cells and DBFS root), standard engine
  (VACUUM gains little from Photon).

```yaml
resources:
  jobs:
    seaco_6161_monthly_vacuum_cleanup:
      name: SEACO-6161 Monthly VACUUM cleanup
      description: >
        Monthly VACUUM of DBFS root Delta tables (dry run). Review the run output, then use
        "Run now with different parameters" with dry_run=false and confirm_workspace=<workspace name>.
      schedule:
        quartz_cron_expression: "0 0 3 28 * ?"  # 03:00 on the 28th of every month
        timezone_id: Europe/Amsterdam
        pause_status: UNPAUSED
      max_concurrent_runs: 1
      timeout_seconds: 21600                     # 6 h hard stop; one VACUUM on millions of small files can take hours
      email_notifications:
        on_success:
          - <your e-mail or team list>           # success = dry run ready for review
        on_failure:
          - <your e-mail or team list>           # failure = error or row_check_failed
      tasks:
        - task_key: Monthly_Vacuum_Cleanup
          notebook_task:
            notebook_path: /Workspace/Users/<user>/databricks-workspace/dbfs-storage-cleanup-SEACO-6161/vacuum_cleanup
            source: WORKSPACE
            base_parameters:
              databases: ""                      # empty = all databases
              vacuum_tables: ""
              exclude_tables: ""
              retain_hours: "168"
              max_tables: "50"
              time_budget_min: "60"
              skip_vacuumed_days: "20"
              apply_recent_vacuum_filter: "true"
              clone_check_scope: all
              dry_run: "true"                    # scheduled run never deletes
              confirm_workspace: ""
          job_cluster_key: Job_cluster
      job_clusters:
        - job_cluster_key: Job_cluster
          new_cluster:
            spark_version: 16.4.x-scala2.12      # LTS, Scala 2.12
            azure_attributes:
              first_on_demand: 1                 # driver on-demand, workers spot
              availability: SPOT_WITH_FALLBACK_AZURE
              spot_bid_max_price: -1
            node_type_id: Standard_D4ds_v5
            enable_elastic_disk: true
            data_security_mode: SINGLE_USER
            runtime_engine: STANDARD
            num_workers: 8
      queue:
        enabled: true
      tags:
        ticket: SEACO-6161
        team: jediknights
```